# Imports and configuration

In [1]:
# ============================================================
# CELL 1 — IMPORTS AND CONFIGURATION
# ============================================================

import os
import gc
import random
from pathlib import Path

import numpy as np
import pandas as pd
import tensorflow as tf
import matplotlib.pyplot as plt

from tensorflow.keras import mixed_precision

print("TensorFlow version:", tf.__version__)

# ------------------------------------------------------------
# Reproducibility
# ------------------------------------------------------------

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

# ------------------------------------------------------------
# Image configuration
# ------------------------------------------------------------

IMG_HEIGHT = 128
IMG_WIDTH = 128

BATCH_SIZE = 16
EPOCHS = 30

AUTOTUNE = tf.data.AUTOTUNE

# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

PROJECT_DIR = Path("/mnt/c/liverCTSegmentation")

SPLIT_DIR = PROJECT_DIR / "outputs" / "splits"
CHECKPOINT_DIR = PROJECT_DIR / "outputs" / "checkpoints"
RESULTS_DIR = PROJECT_DIR / "outputs" / "results"

CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Split directory:", SPLIT_DIR)
print("Checkpoint directory:", CHECKPOINT_DIR)
print("Results directory:", RESULTS_DIR)

# ------------------------------------------------------------
# GPU
# ------------------------------------------------------------

gpus = tf.config.list_physical_devices("GPU")

print("\nGPUs detected:", gpus)

if gpus:
    print("GPU is available.")
else:
    print("WARNING: No GPU detected.")

# ------------------------------------------------------------
# Mixed precision
# ------------------------------------------------------------

mixed_precision.set_global_policy("mixed_float16")

print(
    "Mixed precision policy:",
    mixed_precision.global_policy()
)

2026-10-06 21:48:32.050260: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-10-06 21:48:32.122550: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1791323312.179385  264000 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1791323312.187725  264000 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-10-06 21:48:32.295041: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instr

TensorFlow version: 2.18.0
Project directory: /mnt/c/liverCTSegmentation
Split directory: /mnt/c/liverCTSegmentation/outputs/splits
Checkpoint directory: /mnt/c/liverCTSegmentation/outputs/checkpoints
Results directory: /mnt/c/liverCTSegmentation/outputs/results

GPUs detected: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
GPU is available.
Mixed precision policy: <DTypePolicy "mixed_float16">


# Load the train/validation/test CSVs

In [2]:
# ============================================================
# CELL 2 — LOAD DATA SPLITS
# ============================================================

train_csv = SPLIT_DIR / "train.csv"
val_csv = SPLIT_DIR / "validation.csv"
test_csv = SPLIT_DIR / "test.csv"

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(val_csv)
test_df = pd.read_csv(test_csv)

print("Train samples:", len(train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\nTrain columns:")
print(train_df.columns.tolist())

print("\nFirst training sample:")
print(train_df.iloc[0])

print("\nCSV files loaded successfully.")

Train samples: 39677
Validation samples: 8960
Test samples: 10001

Train columns:
['volume_id', 'image_path', 'mask_path']

First training sample:
volume_id                                            Volume-001
image_path    C:\Users\vivek chauhan\Desktop\dsa babbar\Vide...
mask_path     C:\Users\vivek chauhan\Desktop\dsa babbar\Vide...
Name: 0, dtype: str

CSV files loaded successfully.


# Convert Windows paths to WSL paths

In [3]:
# ============================================================
# CELL 3 — CONVERT WINDOWS PATHS TO WSL PATHS
# ============================================================

def windows_to_wsl(path):
    path = str(path)

    if path.startswith("C:\\"):
        return "/mnt/c/" + path[3:].replace("\\", "/")

    elif path.startswith("C:/"):
        return "/mnt/c/" + path[3:]

    return path


# Apply conversion
for df in [train_df, val_df, test_df]:

    df["image_path"] = df["image_path"].apply(windows_to_wsl)
    df["mask_path"] = df["mask_path"].apply(windows_to_wsl)


# ------------------------------------------------------------
# Verify paths
# ------------------------------------------------------------

print("Example image path:")
print(train_df.iloc[0]["image_path"])

print("\nExample mask path:")
print(train_df.iloc[0]["mask_path"])

print("\nImage exists:",
      Path(train_df.iloc[0]["image_path"]).exists())

print("Mask exists:",
      Path(train_df.iloc[0]["mask_path"]).exists())

Example image path:
/mnt/c/Users/vivek chauhan/Desktop/dsa babbar/Videos/archive (26)/last_dataset/images/Volume-001-000.png

Example mask path:
/mnt/c/Users/vivek chauhan/Desktop/dsa babbar/Videos/archive (26)/last_dataset/masks/Volume-001-000.png

Image exists: True
Mask exists: True


### identifying empty vs non-empty training samples.

Cell 4 — Classify training slices

In [4]:
# ============================================================
# CELL 4 — IDENTIFY EMPTY AND NON-EMPTY TRAINING SLICES
# ============================================================

empty_indices = []
nonempty_indices = []

print("Scanning training masks...")
print("This may take a few minutes.")

for idx, mask_path in enumerate(train_df["mask_path"]):

    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)

    # Count liver pixels
    liver_pixels = tf.reduce_sum(
        tf.cast(mask > 127, tf.int32)
    ).numpy()

    if liver_pixels > 0:
        nonempty_indices.append(idx)
    else:
        empty_indices.append(idx)


print("\n===================================")
print("TRAINING MASK DISTRIBUTION")
print("===================================")

print("Total training slices:", len(train_df))
print("Empty slices:", len(empty_indices))
print("Non-empty slices:", len(nonempty_indices))

print(
    "Empty percentage:",
    100 * len(empty_indices) / len(train_df)
)

print(
    "Non-empty percentage:",
    100 * len(nonempty_indices) / len(train_df)
)

Scanning training masks...
This may take a few minutes.


I0000 00:00:1791323315.537727  264000 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 3539 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 4050 Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.9



TRAINING MASK DISTRIBUTION
Total training slices: 39677
Empty slices: 26589
Non-empty slices: 13088
Empty percentage: 67.01363510346044
Non-empty percentage: 32.986364896539556


# the balanced training DataFrame.

In [22]:
# ============================================================
# CELL 5 — CREATE BALANCED TRAINING DATAFRAME
# ============================================================

# Use the same number of empty and non-empty samples
n_samples_per_class = min(
    len(empty_indices),
    len(nonempty_indices)
)

print("Samples per class:", n_samples_per_class)

# Reproducible random sampling
rng = np.random.default_rng(SEED)

selected_empty = rng.choice(
    empty_indices,
    size=n_samples_per_class,
    replace=False
)

selected_nonempty = rng.choice(
    nonempty_indices,
    size=n_samples_per_class,
    replace=False
)

# Combine both groups
balanced_indices = np.concatenate([
    selected_empty,
    selected_nonempty
])

# Shuffle the combined indices
rng.shuffle(balanced_indices)

# Create balanced DataFrame
balanced_train_df = (
    train_df
    .iloc[balanced_indices]
    .reset_index(drop=True)
)

print("\n===================================")
print("BALANCED TRAINING DATASET")
print("===================================")

print("Original training samples:", len(train_df))
print("Balanced training samples:", len(balanced_train_df))

print("Empty samples:", n_samples_per_class)
print("Non-empty samples:", n_samples_per_class)

print(
    "Empty percentage:",
    100 * n_samples_per_class / len(balanced_train_df)
)

print(
    "Non-empty percentage:",
    100 * n_samples_per_class / len(balanced_train_df)
)

Samples per class: 13088

BALANCED TRAINING DATASET
Original training samples: 39677
Balanced training samples: 26176
Empty samples: 13088
Non-empty samples: 13088
Empty percentage: 50.0
Non-empty percentage: 50.0


In [23]:
# ============================================================
# CELL 6 — IMAGE/MASK LOADING AND TF.DATA PIPELINE
# ============================================================

def load_image_mask(image_path, mask_path):

    # --------------------------------------------------------
    # Read image
    # --------------------------------------------------------
    image = tf.io.read_file(image_path)
    image = tf.image.decode_png(
        image,
        channels=1
    )

    # --------------------------------------------------------
    # Read mask
    # --------------------------------------------------------
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(
        mask,
        channels=1
    )

    # --------------------------------------------------------
    # Resize
    # --------------------------------------------------------
    image = tf.image.resize(
        image,
        [IMG_HEIGHT, IMG_WIDTH],
        method=tf.image.ResizeMethod.BILINEAR
    )

    mask = tf.image.resize(
        mask,
        [IMG_HEIGHT, IMG_WIDTH],
        method=tf.image.ResizeMethod.NEAREST_NEIGHBOR
    )

    # --------------------------------------------------------
    # Normalize image
    # --------------------------------------------------------
    image = tf.cast(
        image,
        tf.float32
    ) / 255.0

    # --------------------------------------------------------
    # Binary mask
    # --------------------------------------------------------
    mask = tf.cast(
        mask > 127,
        tf.float32
    )

    return image, mask


def create_dataset(
    df,
    batch_size=16,
    shuffle=False
):

    dataset = tf.data.Dataset.from_tensor_slices(
        (
            df["image_path"].values,
            df["mask_path"].values
        )
    )

    if shuffle:

        dataset = dataset.shuffle(
            buffer_size=len(df),
            seed=SEED,
            reshuffle_each_iteration=True
        )

    dataset = dataset.map(
        load_image_mask,
        num_parallel_calls=AUTOTUNE
    )

    dataset = dataset.batch(
        batch_size
    )

    dataset = dataset.prefetch(
        AUTOTUNE
    )

    return dataset


print("Dataset functions created successfully.")

Dataset functions created successfully.


In [24]:
# ============================================================
# CELL 7 — CREATE TRAIN / VALIDATION / TEST DATASETS
# ============================================================

# ------------------------------------------------------------
# Balanced training dataset
# ------------------------------------------------------------

train_dataset = create_dataset(
    balanced_train_df,
    batch_size=BATCH_SIZE,
    shuffle=True
)

# ------------------------------------------------------------
# Validation dataset
# IMPORTANT: original validation set is unchanged
# ------------------------------------------------------------

val_dataset = create_dataset(
    val_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)

# ------------------------------------------------------------
# Test dataset
# IMPORTANT: original test set is unchanged
# ------------------------------------------------------------

test_dataset = create_dataset(
    test_df,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("Datasets created successfully.")

print("\nTraining samples:", len(balanced_train_df))
print("Validation samples:", len(val_df))
print("Test samples:", len(test_df))

print("\nBatch size:", BATCH_SIZE)

Datasets created successfully.

Training samples: 26176
Validation samples: 8960
Test samples: 10001

Batch size: 16


In [25]:
# ============================================================
# CELL 8 — VERIFY DATASET BATCH
# ============================================================

# Take one batch from the balanced training dataset
sample_images, sample_masks = next(iter(train_dataset))

print("===================================")
print("DATASET VERIFICATION")
print("===================================")

print("Images shape:", sample_images.shape)
print("Masks shape:", sample_masks.shape)

print("Image dtype:", sample_images.dtype)
print("Mask dtype:", sample_masks.dtype)

print(
    "Image min:",
    tf.reduce_min(sample_images).numpy()
)

print(
    "Image max:",
    tf.reduce_max(sample_images).numpy()
)

print(
    "Mask min:",
    tf.reduce_min(sample_masks).numpy()
)

print(
    "Mask max:",
    tf.reduce_max(sample_masks).numpy()
)

print(
    "Unique mask values:",
    np.unique(sample_masks.numpy())
)

# ------------------------------------------------------------
# Check how many samples in this batch contain liver
# ------------------------------------------------------------

mask_areas = tf.reduce_sum(
    sample_masks,
    axis=[1, 2, 3]
).numpy()

nonempty_in_batch = np.sum(mask_areas > 0)
empty_in_batch = np.sum(mask_areas == 0)

print("\n===================================")
print("BATCH MASK DISTRIBUTION")
print("===================================")

print("Empty samples in batch:", empty_in_batch)
print("Non-empty samples in batch:", nonempty_in_batch)

DATASET VERIFICATION
Images shape: (16, 128, 128, 1)
Masks shape: (16, 128, 128, 1)
Image dtype: <dtype: 'float32'>
Mask dtype: <dtype: 'float32'>
Image min: 0.0
Image max: 1.0
Mask min: 0.0
Mask max: 1.0
Unique mask values: [0. 1.]

BATCH MASK DISTRIBUTION
Empty samples in batch: 9
Non-empty samples in batch: 7


In [35]:
# ============================================================
# CELL 9 — BUILD BASELINE U-NET
# ============================================================

from tensorflow.keras import layers, Model


def conv_block(x, filters):

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu"
    )(x)

    x = layers.Conv2D(
        filters,
        3,
        padding="same",
        activation="relu"
    )(x)

    return x


def build_unet(
    input_shape=(128, 128, 1)
):

    inputs = layers.Input(
        shape=input_shape
    )

    # ========================================================
    # ENCODER
    # ========================================================

    c1 = conv_block(inputs, 64)
    p1 = layers.MaxPooling2D(
        pool_size=(2, 2)
    )(c1)

    c2 = conv_block(p1, 128)
    p2 = layers.MaxPooling2D(
        pool_size=(2, 2)
    )(c2)

    c3 = conv_block(p2, 256)
    p3 = layers.MaxPooling2D(
        pool_size=(2, 2)
    )(c3)

    c4 = conv_block(p3, 512)
    p4 = layers.MaxPooling2D(
        pool_size=(2, 2)
    )(c4)

    # ========================================================
    # BOTTLENECK
    # ========================================================

    c5 = conv_block(p4, 1024)

    # ========================================================
    # DECODER
    # ========================================================

    u6 = layers.Conv2DTranspose(
        512,
        2,
        strides=2,
        padding="same"
    )(c5)

    u6 = layers.Concatenate()(
        [u6, c4]
    )

    c6 = conv_block(u6, 512)

    u7 = layers.Conv2DTranspose(
        256,
        2,
        strides=2,
        padding="same"
    )(c6)

    u7 = layers.Concatenate()(
        [u7, c3]
    )

    c7 = conv_block(u7, 256)

    u8 = layers.Conv2DTranspose(
        128,
        2,
        strides=2,
        padding="same"
    )(c7)

    u8 = layers.Concatenate()(
        [u8, c2]
    )

    c8 = conv_block(u8, 128)

    u9 = layers.Conv2DTranspose(
        64,
        2,
        strides=2,
        padding="same"
    )(c8)

    u9 = layers.Concatenate()(
        [u9, c1]
    )

    c9 = conv_block(u9, 64)

    # ========================================================
    # OUTPUT
    # ========================================================

    outputs = layers.Conv2D(
        1,
        1,
        activation="sigmoid",
        dtype="float32"
    )(c9)

    model = Model(
        inputs=inputs,
        outputs=outputs,
        name="Balanced_UNet"
    )

    return model


model = build_unet()

model.summary()

print(
    "\nTotal parameters:",
    model.count_params()
)

Model: "Balanced_UNet"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 128, 128,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 128, 128,  │        640 │ input_layer[0][0] │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 128,  │     36,928 │ conv2d[0][0]      │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 64, 64,    │          0 │ conv2d_1[0][0]    │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 64,    │     73,856 │ max_pooling2d[0]… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_3 (Conv2D)   │ (None, 64, 64,    │    147,584 │ conv2d_2[0][0]    │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 32, 32,    │          0 │ conv2d_3[0][0]    │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_4 (Conv2D)   │ (None, 32, 32,    │    295,168 │ max_pooling2d_1[… │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_5 (Conv2D)   │ (None, 32, 32,    │    590,080 │ conv2d_4[0][0]    │
│                     │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 16, 16,    │          0 │ conv2d_5[0][0]    │
│ (MaxPooling2D)      │ 256)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_6 (Conv2D)   │ (None, 16, 16,    │  1,180,160 │ max_pooling2d_2[… │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_7 (Conv2D)   │ (None, 16, 16,    │  2,359,808 │ conv2d_6[0][0]    │
│                     │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_3     │ (None, 8, 8, 512) │          0 │ conv2d_7[0][0]    │
│ (MaxPooling2D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_8 (Conv2D)   │ (None, 8, 8,      │  4,719,616 │ max_pooling2d_3[… │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_9 (Conv2D)   │ (None, 8, 8,      │  9,438,208 │ conv2d_8[0][0]    │
│                     │ 1024)             │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_transpose    │ (None, 16, 16,    │  2,097,664 │ conv2d_9[0][0]    │
│ (Conv2DTranspose)   │ 512)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 16, 16,    │          0 │ conv2d_transpose

 Total params: 31,030,593 (118.37 MB)

 Trainable params: 31,030,593 (118.37 MB)

 Non-trainable params: 0 (0.00 B)


Total parameters: 31030593


# Define Dice and BCE + Dice loss

In [36]:
# ============================================================
# CELL 10 — LOSS FUNCTIONS AND DICE METRIC
# ============================================================

def dice_coefficient(y_true, y_pred, smooth=1e-6):

    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)

    y_true = tf.reshape(y_true, [-1])
    y_pred = tf.reshape(y_pred, [-1])

    intersection = tf.reduce_sum(
        y_true * y_pred
    )

    dice = (
        2.0 * intersection + smooth
    ) / (
        tf.reduce_sum(y_true)
        + tf.reduce_sum(y_pred)
        + smooth
    )

    return dice


def dice_loss(y_true, y_pred):

    return 1.0 - dice_coefficient(
        y_true,
        y_pred
    )


def bce_dice_loss(y_true, y_pred):

    bce = tf.keras.losses.binary_crossentropy(
        y_true,
        y_pred
    )

    return tf.reduce_mean(bce) + dice_loss(
        y_true,
        y_pred
    )


print("Loss functions created successfully.")

Loss functions created successfully.


In [37]:
# ============================================================
# CELL 11 — COMPILE BALANCED U-NET
# ============================================================

model.compile(
    optimizer=tf.keras.optimizers.Adam(
        learning_rate=1e-4
    ),
    loss=bce_dice_loss,
    metrics=[
        dice_coefficient
    ]
)

print("Model compiled successfully.")

print("\nOptimizer: Adam")
print("Learning rate: 1e-4")
print("Loss: BCE + Dice")
print("Metric: Dice coefficient")

Model compiled successfully.

Optimizer: Adam
Learning rate: 1e-4
Loss: BCE + Dice
Metric: Dice coefficient


In [38]:
# ============================================================
# CELL 12 — TRAINING CALLBACKS
# ============================================================

best_checkpoint = CHECKPOINT_DIR / "balanced_unet_best.keras"
backup_dir = CHECKPOINT_DIR / "balanced_unet_backup"

callbacks = [
    tf.keras.callbacks.ModelCheckpoint(
        filepath=str(best_checkpoint),
        monitor="val_dice_coefficient",
        mode="max",
        save_best_only=True,
        verbose=1
    ),

    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_dice_coefficient",
        mode="max",
        factor=0.5,
        patience=3,
        min_lr=1e-7,
        verbose=1
    ),

    tf.keras.callbacks.EarlyStopping(
        monitor="val_dice_coefficient",
        mode="max",
        patience=8,
        restore_best_weights=True,
        verbose=1
    ),

    tf.keras.callbacks.BackupAndRestore(
        backup_dir=str(backup_dir)
    )
]

print("Callbacks created successfully.")
print("\nBest checkpoint:")
print(best_checkpoint)

print("\nBackup directory:")
print(backup_dir)

Callbacks created successfully.

Best checkpoint:
/mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras

Backup directory:
/mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_backup


In [39]:
# ============================================================
# CELL 14 — TRAIN BALANCED U-NET
# ============================================================

EPOCHS = 30

print("===================================")
print("STARTING BALANCED U-NET TRAINING")
print("===================================")
print("Training samples:", len(balanced_train_df))
print("Validation samples:", len(val_df))
print("Batch size:", BATCH_SIZE)
print("Maximum epochs:", EPOCHS)
print("Checkpoint:", best_checkpoint)
print()

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=EPOCHS,
    callbacks=callbacks,
    verbose=1
)

print("\n===================================")
print("TRAINING COMPLETED")
print("===================================")
print("Best checkpoint:", best_checkpoint)

STARTING BALANCED U-NET TRAINING
Training samples: 26176
Validation samples: 8960
Batch size: 16
Maximum epochs: 30
Checkpoint: /mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras

Epoch 1/30


/home/vivek_chauhan/tf-env/lib/python3.12/site-packages/keras/src/trainers/epoch_iterator.py:74: UserWarning: `shuffle=True` was passed, but will be ignored since the data `x` was provided as a tf.data.Dataset. The Dataset is expected to already be shuffled (via `.shuffle(buffer_size)`).
  self.data_adapter = data_adapters.get_data_adapter(
I0000 00:00:1791325049.497286  264094 service.cc:148] XLA service 0x7758d800a030 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1791325049.497568  264094 service.cc:156]   StreamExecutor device (0): NVIDIA GeForce RTX 4050 Laptop GPU, Compute Capability 8.9
2026-10-06 22:17:29.659299: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1791325050.162428  264094 cuda_dnn.cc:529] Loaded cuDNN version 90300
2026-10-06 22:17:33.971946: W external/local_xla/xla/tsl/framework/bfc_allocato

1636/1636 ━━━━━━━━━━━━━━━━━━━━ 0s 182ms/step - dice_coefficient: 0.5168 - loss: 0.5558
Epoch 1: val_dice_coefficient improved from None to 0.17676, saving model to /mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras

Epoch 1: finished saving model to /mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 364s 206ms/step - dice_coefficient: 0.5168 - loss: 0.5558 - val_dice_coefficient: 0.1768 - val_loss: 0.8647 - learning_rate: 1.0000e-04
Epoch 2/30
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 0s 190ms/step - dice_coefficient: 0.6147 - loss: 0.4406
Epoch 2: val_dice_coefficient improved from 0.17676 to 0.19194, saving model to /mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras

Epoch 2: finished saving model to /mnt/c/liverCTSegmentation/outputs/checkpoints/balanced_unet_best.keras
1636/1636 ━━━━━━━━━━━━━━━━━━━━ 345s 210ms/step - dice_coefficient: 0.6147 - loss: 0.4406 - val_dice_coefficient: 0.1919 - val_loss: 0

In [40]:
# ============================================================
# CELL 14 — LOAD BEST BALANCED U-NET
# ============================================================

best_balanced_model = tf.keras.models.load_model(
    str(best_checkpoint),
    custom_objects={
        "dice_coefficient": dice_coefficient,
        "bce_dice_loss": bce_dice_loss
    }
)

print("Best Balanced U-Net loaded successfully.")
print("Model name:", best_balanced_model.name)
print("Parameters:", best_balanced_model.count_params())

Best Balanced U-Net loaded successfully.
Model name: Balanced_UNet
Parameters: 31030593


In [42]:
# ============================================================
# CELL 15 — NON-EMPTY SLICE EVALUATION
# ============================================================

import numpy as np


def calculate_metrics(y_true, y_pred, threshold=0.5, smooth=1e-6):

    y_true = np.asarray(y_true).astype(np.float32)
    y_pred = (np.asarray(y_pred) >= threshold).astype(np.float32)

    y_true = y_true.reshape(-1)
    y_pred = y_pred.reshape(-1)

    tp = np.sum(y_true * y_pred)
    tn = np.sum((1 - y_true) * (1 - y_pred))
    fp = np.sum((1 - y_true) * y_pred)
    fn = np.sum(y_true * (1 - y_pred))

    dice = (2 * tp + smooth) / (
        2 * tp + fp + fn + smooth
    )

    iou = (tp + smooth) / (
        tp + fp + fn + smooth
    )

    precision = (tp + smooth) / (
        tp + fp + smooth
    )

    recall = (tp + smooth) / (
        tp + fn + smooth
    )

    accuracy = (tp + tn + smooth) / (
        tp + tn + fp + fn + smooth
    )

    return dice, iou, precision, recall, accuracy


def evaluate_nonempty(model, dataset):

    dice_scores = []
    iou_scores = []
    precision_scores = []
    recall_scores = []
    accuracy_scores = []

    total = 0
    nonempty = 0
    empty = 0

    for images, masks in dataset:

        predictions = model.predict(
            images,
            verbose=0
        )

        for i in range(images.shape[0]):

            gt = masks[i].numpy()
            pred = predictions[i]

            # Check whether ground-truth contains liver
            if np.sum(gt) > 0:

                nonempty += 1

                dice, iou, precision, recall, accuracy = calculate_metrics(
                    gt,
                    pred
                )

                dice_scores.append(dice)
                iou_scores.append(iou)
                precision_scores.append(precision)
                recall_scores.append(recall)
                accuracy_scores.append(accuracy)

            else:
                empty += 1

            total += 1

    results = {
        "total_slices": total,
        "nonempty_slices": nonempty,
        "empty_slices": empty,
        "dice": np.mean(dice_scores),
        "iou": np.mean(iou_scores),
        "precision": np.mean(precision_scores),
        "recall": np.mean(recall_scores),
        "accuracy": np.mean(accuracy_scores)
    }

    return results


# ============================================================
# VALIDATION
# ============================================================

print("Evaluating validation set...")

balanced_val_results = evaluate_nonempty(
    best_balanced_model,
    val_dataset
)

print("\nBalanced U-Net — Validation (Non-empty slices)")
print("------------------------------------------------")

for key, value in balanced_val_results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")


# ============================================================
# TEST
# ============================================================

print("\nEvaluating test set...")

balanced_test_results = evaluate_nonempty(
    best_balanced_model,
    test_dataset
)

print("\nBalanced U-Net — Test (Non-empty slices)")
print("------------------------------------------")

for key, value in balanced_test_results.items():

    if isinstance(value, float):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

Evaluating validation set...


2026-10-07 10:23:41.644156: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence



Balanced U-Net — Validation (Non-empty slices)
------------------------------------------------
total_slices: 8960
nonempty_slices: 2819
empty_slices: 6141
dice: 0.6070522665977478
iou: 0.5386465191841125
precision: 0.7502378225326538
recall: 0.6823501586914062
accuracy: 0.9741289019584656

Evaluating test set...

Balanced U-Net — Test (Non-empty slices)
------------------------------------------
total_slices: 10001
nonempty_slices: 3216
empty_slices: 6785
dice: 0.46413058042526245
iou: 0.405945360660553
precision: 0.642547607421875
recall: 0.5873469114303589
accuracy: 0.9701321125030518


2026-10-07 10:24:55.767395: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
